In [1]:
# Machine learning

### Imports

In [2]:
# Import packages
from pathlib import Path
import pandas as pd
import numpy as np

from scipy.stats import zscore
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [3]:
FOLDERS = {
    "../data/03_hammer_tests/test_01/00_intact": "No fracture hammer 1",
    "../data/03_hammer_tests/test_01/100_fractured": "100% Fracture hammer 1",
    "../data/03_hammer_tests/test_02/00_intact": "No fracture hammer 2",
    "../data/03_hammer_tests/test_02/100_fractured": "100% Fracture hammer 2",
    "../data/03_hammer_tests/test_03/00_intact": "No fracture hammer 3",
    "../data/03_hammer_tests/test_03/100_fractured": "100% Fracture hammer 3",
    "../data/03_hammer_tests/test_04/00_intact": "No fracture hammer 4",
    "../data/03_hammer_tests/test_04/100_fractured": "100% Fracture hammer 4"
}

group_data = {}

for folder, label in FOLDERS.items():

    csv_files = sorted(Path(folder).glob("*.csv"))

    if not csv_files:
        print(f"No CSV files found in {folder}")
        continue

    impedances = []
    frequency = None

    for csv_file in csv_files:

        df = pd.read_csv(csv_file)

        if frequency is None:
            frequency = df["frequency"].values

        # Skip incomplete sweeps
        if len(df) != len(frequency):
            print(
                f"Skipping {csv_file.name}: "
                f"{len(df)} rows instead of {len(frequency)}"
            )
            continue

        impedances.append(df["impedance"].values)

    if len(impedances) == 0:
        print(f"No valid files found in {folder}")
        continue

    impedances = np.vstack(impedances)

    group_data[label] = {
        "frequency": frequency,
        "mean": np.mean(impedances, axis=0),
        "std": np.std(impedances, axis=0),
        "raw": impedances,
        "n": len(csv_files)
    }

    print(f"{label}: {len(impedances)} files loaded")

No fracture hammer 1: 6 files loaded
100% Fracture hammer 1: 15 files loaded
No fracture hammer 2: 11 files loaded
100% Fracture hammer 2: 14 files loaded
No fracture hammer 3: 7 files loaded
100% Fracture hammer 3: 7 files loaded
No fracture hammer 4: 9 files loaded
100% Fracture hammer 4: 9 files loaded


### Outlier detection

In [4]:
print("OUTLIER REPORT")
print("=" * 40)

for group_name, data in group_data.items():

    sweep_means = np.mean(
        data["raw"],
        axis=1
    )

    z_scores = np.abs(
        zscore(sweep_means)
    )

    outliers = np.where(
        z_scores > 2.5
    )[0]

    print(f"\n{group_name}")

    if len(outliers) == 0:
        print("No outliers found")

    else:
        print(f"Outliers: {outliers}")

OUTLIER REPORT

No fracture hammer 1
No outliers found

100% Fracture hammer 1
No outliers found

No fracture hammer 2
Outliers: [3]

100% Fracture hammer 2
Outliers: [9]

No fracture hammer 3
No outliers found

100% Fracture hammer 3
No outliers found

No fracture hammer 4
Outliers: [0]

100% Fracture hammer 4
No outliers found


In [5]:
import pandas as pd
import numpy as np
from itertools import combinations

rows = []

for hammer in [1, 2, 3, 4]:

    intact_key = f"No fracture hammer {hammer}"
    fracture_key = f"100% Fracture hammer {hammer}"

    intact_sweeps = group_data[intact_key]["raw"]
    fracture_sweeps = group_data[fracture_key]["raw"]

    # Fracture examples
    for intact in intact_sweeps:
        for fractured in fracture_sweeps:

            relative_change = (
                fractured - intact
            ) / intact

            row = dict(zip(freq, relative_change))
            row["broken"] = True
            row["hammer"] = hammer

            rows.append(row)

    # Non-fracture examples
    for intact1, intact2 in combinations(intact_sweeps, 2):

        relative_change = (
            intact2 - intact1
        ) / intact1

        row = dict(zip(freq, relative_change))
        row["broken"] = False
        row["hammer"] = hammer

        rows.append(row)


df_changes = pd.DataFrame(rows)

print(df_changes.shape)
display(df_changes.head())

NameError: name 'freq' is not defined

In [ ]:
df_changes

In [ ]:
X = df_changes.drop(columns=["broken", "hammer"])
y = df_changes["broken"]

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression())
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

print("Fold accuracies:", scores)
print("Mean accuracy:", scores.mean())
print("Std:", scores.std())

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95)),
    ("classifier", LogisticRegression())
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

print("Fold accuracies:", scores)
print("Mean accuracy:", scores.mean())
print("Std:", scores.std())

In [ ]:
print(group_data.keys())

first_key = list(group_data.keys())[0]

print(group_data[first_key].keys())

### test on 1-3, train on 4

In [ ]:
train_df = df_changes[df_changes["hammer"].isin([1, 2, 3])]

test_df = df_changes[df_changes["hammer"] == 4]

X_train = train_df.drop(columns=["broken", "hammer"])
y_train = train_df["broken"]

X_test = test_df.drop(columns=["broken", "hammer"])
y_test = test_df["broken"]

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
confusion_matrix(y_test, y_pred)